In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import h5py 
import sys
import data_analysis.mea_analysis as mea
from tqdm import tqdm
from pathlib import Path
import json
import cv2

In [9]:
BRW_BASE_FOLDER = Path("/media/ferdinand-forberger/RawDataBackUp")
MAIN_FOLDER = Path("/media/ferdinand-forberger/Seagate Portable Drive")
GRAPH_METRICS_PATH = MAIN_FOLDER / "graph_metrics.xlsx"



df_overview = mea.get_overview(MAIN_FOLDER)
df_overview = df_overview[df_overview["sles_saved"]==True]
df_overview.reset_index(inplace=True, drop=True)
df_overview["image_path"]= df_overview["folder_path"].astype(str) + "/org_image.png"
df_overview["mask_path"] = df_overview["folder_path"].astype(str) + "/mask.png"

#get metadata file to have info about which slice belongs to which group
df_metadata = pd.read_excel("/media/ferdinand-forberger/RawDataBackUp/meta_data_file.xlsx")
df_metadata["folder"] = df_metadata["animal_number"].astype(str) + "_" + df_metadata["slice_number"].astype(str)
df_overview = pd.merge(df_overview, df_metadata, left_on="folder", right_on="folder", how="left")
df_overview["sttc_path"] = df_overview["folder_path"] / "sttc.npy"

In [10]:
def find_rot_json(folder_path : Path):
    json_file = list(folder_path.glob("*transform.json"))
    return json_file[0] if len(json_file)>0 else None
path = df_overview["folder_path"].iloc[0]
json_paths = df_overview["folder_path"].apply(find_rot_json)
df_overview["json_path"] = json_paths

In [11]:
def transform_matrix_from_json(data_matrix: np.ndarray, json_path: str, original_image: np.ndarray) -> np.ndarray:

    if not isinstance(data_matrix, np.ndarray) or data_matrix.ndim not in [2, 3]:
        raise ValueError("data_matrix must be a 2D or 3D numpy array.")
    
    if data_matrix.shape[0] != data_matrix.shape[1]:
        raise ValueError("The first two dimensions (height and width) of data_matrix must be equal.")

    with open(json_path, 'r') as f:
        params = json.load(f)

    h_data, w_data = data_matrix.shape[:2]
    h_orig, w_orig = original_image.shape[:2]

    scale_x = w_data / w_orig if w_orig > 0 else 1
    scale_y = h_data / h_orig if h_orig > 0 else 1
    
    angle = params['angle_degrees']
    scaled_dx = params['dx_pixels'] * scale_x
    scaled_dy = params['dy_pixels'] * scale_y

    center = (w_data / 2, h_data / 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    M[0, 2] += scaled_dx
    M[1, 2] += scaled_dy

    warped_matrix = cv2.warpAffine(
        data_matrix,
        M,
        (w_data, h_data),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    return warped_matrix

def unit_to_amplitude(unit, units, amplitudes):
    mask = units == unit
    mean_apl = np.mean(amplitudes[mask])
    return mean_apl if not np.isnan(mean_apl) else 0

,clu,num_spikes,chan_best,mua_labels,freq,amplitude
138,138,3002,2768,good,5.003333,18.963112
228,228,2889,2131,good,4.815000,18.068747
267,267,2776,1671,good,4.626667,22.338114
195,195,2648,2323,good,4.413333,23.527992
191,191,2582,2377,good,4.303333,34.569321
242,242,2043,1993,good,3.405000,30.641155
132,132,1868,2850,good,3.113333,38.946140
174,174,1858,2442,good,3.096667,36.477554
225,225,1836,2059,good,3.060000,29.599771
320,320,1665,1201,good,2.775000,17.418791


In [29]:
aligned_maps = {"Freq" : [],
                 "Ampl" : [],
                 "Origins" : []}
max_freqs = []
max_ampls = []

for i, row in tqdm(df_overview.iterrows(), total=len(df_overview)):
    json_path = row["json_path"]
    st_df, st, clu, best_cha_st, vectorized_map, amplitudes= mea.get_stdf(parent_folder_path=row["folder_path"],
                                                                    return_amplitudes=True)
    st_df = st_df[st_df["mua_labels"]=="good"]
    st_df = st_df.loc[st_df["num_spikes"] > 100]
    st_df["freq"] = st_df["num_spikes"] / 600
    max_freqs.append(st_df['freq'].max())



    st_df["amplitude"] = st_df["clu"].apply(lambda u: unit_to_amplitude(u, clu, amplitudes))
    max_ampls.append(st_df['amplitude'].max())
    st_df_temp = st_df.drop_duplicates(subset=["chan_best"], keep="first")

    current_max_freq = st_df['freq'].max()
    current_max_ampl = st_df['amplitude'].max()

    org_img = plt.imread(row["image_path"])
    chan_map = np.arange(4096).reshape(64,64)
    chan_map_rotated = transform_matrix_from_json(chan_map, 
                                                json_path, 
                                                original_image=org_img)

    # maps
    freq_map = np.zeros(4096)
    for _, sub_row in st_df_temp.iterrows():
        chan = sub_row["chan_best"]
        freq_map[chan] = sub_row["freq"]
    freq_map = freq_map.reshape(64,64)

    ampl_map = np.zeros(4096)
    for _, sub_row in st_df_temp.iterrows():
        chan = sub_row["chan_best"]
        ampl_map[chan] = sub_row["amplitude"]
    ampl_map = ampl_map.reshape(64,64)

    freq_map_rotated = transform_matrix_from_json(freq_map, 
                                                json_path, 
                                                original_image=org_img)
    ampl_map_rotated = transform_matrix_from_json(ampl_map, 
                                                json_path, 
                                                original_image=org_img)
    
    freq_map_rotated[freq_map_rotated==0] = np.nan
    ampl_map_rotated[ampl_map_rotated==0] = np.nan


    non_nan_max_freq = np.nanmax(max_freqs)

    epsilon = 1e-9 

    assert np.nanmax(freq_map_rotated) <= current_max_freq + epsilon, \
        f"Max freq validation failed: Transformed max ({np.nanmax(freq_map_rotated)}) > Original max ({current_max_freq})"

    assert np.nanmax(ampl_map_rotated) <= current_max_ampl + epsilon, \
        f"Max ampl validation failed: Transformed max ({np.nanmax(ampl_map_rotated)}) > Original max ({current_max_ampl})"

    aligned_maps["Freq"].append(freq_map_rotated)
    aligned_maps["Ampl"].append(ampl_map_rotated)

    #origins
    burst_json_path = Path(row["folder_path"]) / "burst_params.json"
    NBs = np.load(row["sle_path"])
    res = mea.filter_spikes_find_origin(sles=NBs,
                                                st=st,
                                                clu=clu,
                                                cluster_to_channel_map=vectorized_map, 
                                                burst_json_path=burst_json_path, 
                                                folder_path=row["folder_path"])
    estimated_SLE_orgins, first_spikes, unique_chans_list, filtered_first_spikes, filtered_chans, chan_subsets = res
    
    origins_map = np.zeros(4096)
    estimated_SLE_orgins = np.round(estimated_SLE_orgins).astype(int)
    channels = mea.coords_to_chans(x_coords=estimated_SLE_orgins[:,0], y_coords=estimated_SLE_orgins[:,1])
    for channel in channels:
        
        origins_map[channel] += 1
    origins_map = origins_map.reshape(64,64)

    origins_map_rotated = transform_matrix_from_json(origins_map, 
                                                json_path,
                                                original_image=org_img)
    origins_map_rotated[origins_map_rotated==0] = np.nan

    aligned_maps["Origins"].append(origins_map_rotated)

aligned_maps["Freq"] = np.array(aligned_maps["Freq"])
assert np.all(np.nan_to_num(aligned_maps["Freq"]) <= np.max(max_freqs))

aligned_maps["Ampl"] = np.array(aligned_maps["Ampl"])
assert np.all(np.nan_to_num(aligned_maps["Ampl"]) <= np.max(max_ampls))

aligned_maps["Origins"] = np.array(aligned_maps["Origins"])


  0%|          | 0/22 [00:00<?, ?it/s]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 408, disagreement: 0
Number of good clusters (Original): 121
Number of good clusters (Changed): 121
-------------------------------------------------------
Comparison of Labels (Original vs. Changed)
KSLabel agreement: 408, disagreement: 0
Number of good clusters (Original): 121
Number of good clusters (Changed): 121
-------------------------------------------------------


  5%|▍         | 1/22 [00:02<00:48,  2.29s/it]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 554, disagreement: 0
Number of good clusters (Original): 119
Number of good clusters (Changed): 119
-------------------------------------------------------
Comparison of Labels (Original vs. Changed)
KSLabel agreement: 554, disagreement: 0
Number of good clusters (Original): 119
Number of good clusters (Changed): 119
-------------------------------------------------------


  9%|▉         | 2/22 [00:06<01:10,  3.53s/it]

...

 91%|█████████ | 20/22 [02:06<00:14,  7.48s/it]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 450, disagreement: 0
Number of good clusters (Original): 91
Number of good clusters (Changed): 91
-------------------------------------------------------
Comparison of Labels (Original vs. Changed)
KSLabel agreement: 450, disagreement: 0
Number of good clusters (Original): 91
Number of good clusters (Changed): 91
-------------------------------------------------------


 95%|█████████▌| 21/22 [02:12<00:07,  7.23s/it]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 323, disagreement: 0
Number of good clusters (Original): 55
Number of good clusters (Changed): 55
-------------------------------------------------------
Comparison of Labels (Original vs. Changed)
KSLabel agreement: 323, disagreement: 0
Number of good clusters (Original): 55
Number of good clusters (Changed): 55
-------------------------------------------------------


100%|██████████| 22/22 [02:17<00:00,  6.25s/it]


In [31]:
save_folder = Path("data")

np.save(save_folder / "aligned_freq_maps.npy", aligned_maps["Freq"])
np.save(save_folder / "aligned_ampl_maps.npy", aligned_maps["Ampl"])
np.save(save_folder / "aligned_origins_maps.npy", aligned_maps["Origins"])